In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import joblib
import glob

csv_files=sorted(glob.glob('/content/sample_data/ppg_data_*.csv'))
dfs=[]
for i, f in enumerate(csv_files):
    d = pd.read_csv(f)
    d['person_id']=i
    dfs.append(d)

df=pd.concat(dfs, ignore_index=True)
df=df[df['status']=='NORMAL'].copy()
df=df.dropna(subset=['hr', 'spo2'])
df=df[(df['hr']>40) & (df['hr']<200)]
df=df[(df['spo2']>80) & (df['spo2']<=100)]
print(f"Total samples: {len(df)}, people: {df['person_id'].nunique()}")

LOGS=50
HORIZON=150

def make_features(hr_win, spo2_win):
    hr_m=hr_win.mean()
    spo2_m=spo2_win.mean()
    def safe_corr(w):
        if w.std()==0:
          return 0.0
        r=np.corrcoef(w[:-1], w[1:])[0, 1]
        return float(r) if np.isfinite(r) else 0.0
    return (
        list(hr_win-hr_m) + list(spo2_win-spo2_m) +
        [
            hr_m,         hr_win.std(),
            float(hr_win.max()-hr_win.min()),
            float(hr_win[-1]-hr_win[0]),
            float(hr_win[-1]-hr_win[-10]),
            float(hr_win[-1]-hr_win[-25]),
            float(np.polyfit(np.arange(LOGS), hr_win, 1)[0]),
            safe_corr(hr_win),
            spo2_m, spo2_win.std(),
            float(spo2_win.max()-spo2_win.min()),
            float(spo2_win[-1]-spo2_win[0]),
            float(spo2_win[-1]-spo2_win[-10]),
            float(spo2_win[-1]-spo2_win[-25]),
            float(np.polyfit(np.arange(LOGS), spo2_win, 1)[0]),
            safe_corr(spo2_win),
        ]
    )

#Build features per-person to avoid window crossing person boundaries
all_X, all_Y, all_person=[], [], []

for pid, group in df.groupby('person_id'):
    hr=group['hr'].values.astype(float)
    spo2=group['spo2'].values.astype(float)
    for i in range(LOGS, len(hr) - HORIZON):
        hr_win=hr[i-LOGS:i]
        spo2_win=spo2[i-LOGS:i]
        all_X.append(make_features(hr_win, spo2_win))
        all_Y.append([hr[i+HORIZON]-hr_win[-1],
                      spo2[i+HORIZON]-spo2_win[-1]])
        all_person.append(pid)

X=np.array(all_X)
Y=np.array(all_Y)
persons=np.array(all_person)

#Drop NaNs
mask=~(np.isnan(X).any(axis=1) | np.isnan(Y).any(axis=1))
X, Y, persons=X[mask], Y[mask], persons[mask]
print(f"Samples: {len(X)}, features: {X.shape[1]}")

#Leave-One-Person-Out CV, to see if the model can generalize to a new person
print("\nLeave-One-Person-Out CV:")
hr_r2s, spo2_r2s=[], []
unique_persons = np.unique(persons)

for pid in unique_persons:
    test_mask=persons==pid
    train_mask=~test_mask
    scaler_cv=StandardScaler()
    X_tr=scaler_cv.fit_transform(X[train_mask])
    X_te=scaler_cv.transform(X[test_mask])
    m=MultiOutputRegressor(
        GradientBoostingRegressor(n_estimators=400, max_depth=3,
                                   learning_rate=0.04, subsample=0.8,
                                   min_samples_leaf=4, random_state=42),
        n_jobs=-1)
    m.fit(X_tr, Y[train_mask])
    p=m.predict(X_te)
    hr_r2=r2_score(Y[test_mask, 0], p[:, 0])
    spo2_r2=r2_score(Y[test_mask, 1], p[:, 1])
    hr_r2s.append(hr_r2)
    spo2_r2s.append(spo2_r2)
    n_test=test_mask.sum()
    print(f"  Person {pid} (n={n_test:4d}): HR R²={hr_r2:.3f}  SpO2 R²={spo2_r2:.3f}")

print(f"\nLOPO mean — HR R²={np.mean(hr_r2s):.3f}±{np.std(hr_r2s):.3f}  "
      f"SpO2 R²={np.mean(spo2_r2s):.3f}±{np.std(spo2_r2s):.3f}")

#Final model on all data
scaler_final=StandardScaler()
X_all=scaler_final.fit_transform(X)
model_final=MultiOutputRegressor(
    GradientBoostingRegressor(n_estimators=400, max_depth=3,
                               learning_rate=0.04, subsample=0.8,
                               min_samples_leaf=4, random_state=42),
    n_jobs=-1)
model_final.fit(X_all, Y)

Y_pred=model_final.predict(X_all)
print(f"\nFull-data MAE: HR={np.mean(np.abs(Y[:,0]-Y_pred[:,0])):.1f} bpm  "
      f"SpO2={np.mean(np.abs(Y[:,1]-Y_pred[:,1])):.2f}%")

joblib.dump({
    'model':      model_final,
    'scaler':     scaler_final,
    'n_features': X.shape[1],
    'logs':       LOGS,
    'horizon':    HORIZON,
    'mode':       'delta',
}, 'predict_short_multi.pkl')
print("Saved predict_short_multi.pkl")

Total samples: 3349, people: 6


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Samples: 2196, features: 116

Leave-One-Person-Out CV:
  Person 0 (n= 716): HR R²=0.416  SpO2 R²=0.367
  Person 1 (n= 449): HR R²=0.399  SpO2 R²=-2.360
  Person 3 (n= 227): HR R²=0.622  SpO2 R²=0.055
  Person 4 (n= 401): HR R²=0.248  SpO2 R²=-0.369
  Person 5 (n= 403): HR R²=0.401  SpO2 R²=0.298

LOPO mean — HR R²=0.417±0.119  SpO2 R²=-0.402±1.012

Full-data MAE: HR=1.5 bpm  SpO2=0.33%
Saved predict_short_multi.pkl
